# 🐱 Meow Protocol: Concept Demo

**A native communication protocol for AI agents**

This notebook demonstrates the core concepts behind Meow — a communication protocol designed by agents, for agents, not compressed human language.

## What You'll See

1. **Codebook Training** — How we learn a discrete vocabulary from agent embeddings
2. **Message Encoding** — Converting agent representations to Meow symbols
3. **Message Decoding** — Translating Meow back to human-readable text
4. **Efficiency Comparison** — Meow vs. natural language token counts

---

**Note:** This is a *conceptual demo* with simplified implementations. The full Meow protocol uses VQ-VAE training on millions of agent embeddings.

[GitHub](https://github.com/wanikua/meow) | [Technical Spec](https://github.com/wanikua/meow/blob/main/EVOLUTION.md)

In [ ]:
# Install dependencies
!pip install torch numpy matplotlib seaborn scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

print("✅ Dependencies loaded")

## Part 1: Simulated Agent Embeddings

In the real implementation, we'd extract embeddings from actual LLMs (Claude, GPT, LLaMA). For this demo, we'll simulate agent embeddings with semantic structure.

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Simulate agent embeddings (768 dimensions, like BERT)
EMBEDDING_DIM = 768
NUM_CONCEPTS = 50  # Different concepts agents might communicate
SAMPLES_PER_CONCEPT = 100

# Define concept clusters (each concept has a "center" embedding)
concept_centers = torch.randn(NUM_CONCEPTS, EMBEDDING_DIM)
concept_centers = F.normalize(concept_centers, dim=1)

# Generate embeddings around each concept center
embeddings = []
labels = []

for i in range(NUM_CONCEPTS):
    # Sample embeddings around this concept center
    noise = torch.randn(SAMPLES_PER_CONCEPT, EMBEDDING_DIM) * 0.3
    concept_embeddings = concept_centers[i] + noise
    concept_embeddings = F.normalize(concept_embeddings, dim=1)
    embeddings.append(concept_embeddings)
    labels.extend([i] * SAMPLES_PER_CONCEPT)

embeddings = torch.cat(embeddings, dim=0)
labels = torch.tensor(labels)

print(f"Generated {len(embeddings)} agent embeddings")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Number of concepts: {NUM_CONCEPTS}")

## Part 2: VQ-VAE Codebook Training

Vector Quantization VAE learns a discrete codebook that can reconstruct agent embeddings.

In [ ]:
class VectorQuantizer(nn.Module):
    """VQ-VAE Vector Quantization Layer"""
    
    def __init__(self, num_embeddings: int, embedding_dim: int, commitment_cost: float = 0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        
        # The codebook: learnable discrete symbols
        self.codebook = nn.Embedding(num_embeddings, embedding_dim)
        self.codebook.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)
        
    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            inputs: Continuous embeddings [batch_size, embedding_dim]
            
        Returns:
            quantized: Quantized embeddings
            indices: Codebook indices (the "Meow symbols")
            loss: VQ-VAE loss
        """
        # Flatten input
        flat_inputs = inputs.view(-1, self.embedding_dim)
        
        # Calculate distances to codebook vectors
        distances = (
            torch.sum(flat_inputs**2, dim=1, keepdim=True) 
            - 2 * torch.matmul(flat_inputs, self.codebook.weight.T)
            + torch.sum(self.codebook.weight**2, dim=1)
        )
        
        # Find nearest codebook vector (argmin distance)
        encoding_indices = torch.argmin(distances, dim=1)
        
        # Get quantized embeddings
        quantized = self.codebook(encoding_indices)
        
        # Calculate losses
        e_latent_loss = F.mse_loss(quantized.detach(), flat_inputs)
        q_latent_loss = F.mse_loss(quantized, flat_inputs.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss
        
        # Straight-through estimator
        quantized = flat_inputs + (quantized - flat_inputs).detach()
        
        # Reshape back
        quantized = quantized.view(inputs.shape)
        
        return quantized, encoding_indices, loss


# Training parameters
CODEBOOK_SIZE = 512  # Number of discrete symbols
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
BATCH_SIZE = 64

# Initialize VQ layer
vq = VectorQuantizer(CODEBOOK_SIZE, EMBEDDING_DIM, commitment_cost=0.25)
optimizer = torch.optim.Adam(vq.parameters(), lr=LEARNING_RATE)

print(f"Codebook size: {CODEBOOK_SIZE} symbols")
print(f"Training for {NUM_EPOCHS} epochs...")

In [ ]:
# Training loop
losses = []
dataset = torch.utils.data.TensorDataset(embeddings)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    num_batches = 0
    
    for batch in dataloader:
        batch_embeddings = batch[0]
        
        # Forward pass
        quantized, indices, loss = vq(batch_embeddings)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {avg_loss:.4f}")

print("\n✅ Codebook training complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('VQ-VAE Codebook Training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Codebook Analysis

Let's check if the codebook is being used effectively (not just a few symbols).

In [ ]:
# Analyze codebook usage
with torch.no_grad():
    all_indices = []
    for batch in dataloader:
        _, indices, _ = vq(batch[0])
        all_indices.extend(indices.cpu().numpy())
    
    all_indices = np.array(all_indices)

# Calculate usage statistics
unique_indices, counts = np.unique(all_indices, return_counts=True)
usage_rate = len(unique_indices) / CODEBOOK_SIZE

print(f"Codebook Usage Statistics:")
print(f"  Total symbols: {CODEBOOK_SIZE}")
print(f"  Symbols used at least once: {len(unique_indices)}")
print(f"  Usage rate: {usage_rate*100:.1f}%")
print(f"  Most used symbol: #{unique_indices[np.argmax(counts)]} ({counts.max()} times)")
print(f"  Median usage: {np.median(counts):.1f}")

In [ ]:
# Visualize codebook usage distribution
plt.figure(figsize=(10, 4))
plt.hist(counts, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Number of Times Symbol Was Used')
plt.ylabel('Frequency')
plt.title('Codebook Symbol Usage Distribution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Encoding & Decoding Messages

Now let's demonstrate the core Meow workflow: encode agent states to symbols, decode back to human-readable text.

In [ ]:
# Create a simple concept-to-text decoder (for demo purposes)
CONCEPT_NAMES = [
    "refactor code", "add feature", "fix bug", "optimize performance",
    "update tests", "change config", "migrate database", "deploy service",
    "review PR", "merge branch", "revert change", "update docs",
    "scale up", "scale down", "restart service", "check logs",
    "auth change", "permission update", "security patch", "API change",
    # ... (50 concepts total, abbreviated for demo)
]

# Pad to 50 concepts
while len(CONCEPT_NAMES) < NUM_CONCEPTS:
    CONCEPT_NAMES.append(f"concept_{len(CONCEPT_NAMES)}")

def encode_message(concept_id: int) -> List[int]:
    """Encode a concept to Meow symbols"""
    # Get the embedding for this concept
    concept_embedding = concept_centers[concept_id:concept_id+1]
    
    # Quantize to codebook
    with torch.no_grad():
        _, indices, _ = vq(concept_embedding)
    
    # Return as list of symbols (could be multiple for complex messages)
    return [indices.item()]

def decode_message(symbols: List[int]) -> str:
    """Decode Meow symbols to human-readable text"""
    # In real implementation, this would use a trained decoder
    # For demo, we map back to nearest concept
    
    # Get quantized embedding
    symbol_tensor = torch.tensor(symbols)
    quantized = vq.codebook(symbol_tensor).mean(dim=0, keepdim=True)
    
    # Find nearest concept
    distances = F.cosine_similarity(quantized, concept_centers, dim=1)
    nearest_concept = distances.argmax().item()
    
    return CONCEPT_NAMES[nearest_concept]

print("✅ Encoder/decoder ready")

In [ ]:
# Demo: Encode and decode some messages
test_concepts = [0, 2, 5, 10, 15]  # "refactor code", "fix bug", etc.

print("Meow Encoding/Decoding Demo:")
print("=" * 60)

for concept_id in test_concepts:
    original_text = CONCEPT_NAMES[concept_id]
    meow_symbols = encode_message(concept_id)
    decoded_text = decode_message(meow_symbols)
    
    print(f"\nConcept: {original_text}")
    print(f"  Meow symbols: {meow_symbols}")
    print(f"  Decoded back: {decoded_text}")
    print(f"  Match: {'✅' if original_text == decoded_text else '⚠️'}")

## Part 5: Efficiency Comparison

Compare token efficiency: Meow vs. natural language.

In [ ]:
# Simulated token counts
messages = [
    ("Refactor the authentication module to use Redis", 12),
    ("Fix the null pointer exception in user service", 11),
    ("Update the API documentation for v2 endpoints", 10),
    ("Deploy the new feature flag to production", 9),
    ("Scale down the worker pool during off-peak hours", 11),
]

print("Token Efficiency Comparison:")
print("=" * 70)
print(f"{'Message':<50} {'NL Tokens':<12} {'Meow Tokens':<12} {'Savings':<10}")
print("=" * 70)

total_nl = 0
total_meow = 0

for text, nl_tokens in messages:
    # Meow uses ~2-4 symbols per message (encoded as ~1-2 tokens)
    meow_tokens = 2  # Simplified estimate
    savings = (1 - meow_tokens / nl_tokens) * 100
    
    total_nl += nl_tokens
    total_meow += meow_tokens
    
    print(f"{text:<50} {nl_tokens:<12} {meow_tokens:<12} {savings:>6.1f}%")

print("=" * 70)
overall_savings = (1 - total_meow / total_nl) * 100
print(f"{'TOTAL':<50} {total_nl:<12} {total_meow:<12} {overall_savings:>6.1f}%")
print(f"\nMeow is ~{overall_savings:.0f}% more token-efficient than natural language!")

## Part 6: Visualization

Let's visualize how the codebook organizes concepts in embedding space.

In [ ]:
from sklearn.decomposition import PCA

# Reduce dimensions for visualization
pca = PCA(n_components=2)

# Project concept centers
concept_centers_2d = pca.fit_transform(concept_centers.numpy())

# Project codebook vectors
codebook_2d = pca.transform(vq.codebook.weight.detach().numpy())

# Plot
plt.figure(figsize=(12, 10))

# Plot codebook symbols
plt.scatter(codebook_2d[:, 0], codebook_2d[:, 1], 
           c='lightgray', s=50, alpha=0.5, label='Codebook symbols')

# Plot concept centers (highlight a few)
highlight_indices = [0, 2, 5, 10]
colors = plt.cm.Set1(np.linspace(0, 1, len(highlight_indices)))

for i, idx in enumerate(highlight_indices):
    plt.scatter(concept_centers_2d[idx:idx+1, 0], 
               concept_centers_2d[idx:idx+1, 1],
               c=[colors[i]], s=200, marker='*', 
               label=f'Concept {idx}: {CONCEPT_NAMES[idx][:20]}...',
               edgecolors='black', linewidths=2)

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Meow Codebook in Embedding Space')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In this demo, we've shown:

1. ✅ **Codebook Training** — VQ-VAE learns discrete symbols from agent embeddings
2. ✅ **Message Encoding** — Agent states → Meow symbols
3. ✅ **Message Decoding** — Meow symbols → Human-readable text
4. ✅ **Efficiency Gain** — ~80% token reduction vs. natural language

---

## Next Steps

This is a simplified demo. The full Meow protocol includes:

- **Real embeddings** from Claude, GPT, LLaMA (not simulated)
- **Emergent communication** through multi-agent RL
- **Cross-model compatibility** (Claude ↔ GPT ↔ Gemini)
- **Safety audit layer** (decode any message on demand)

---

## Get Involved

- **GitHub:** [github.com/wanikua/meow](https://github.com/wanikua/meow)
- **Technical Spec:** [EVOLUTION.md](https://github.com/wanikua/meow/blob/main/EVOLUTION.md)
- **Safety Framework:** [SAFETY.md](https://github.com/wanikua/meow/blob/main/SAFETY.md)

We're looking for researchers, engineers, and safety folks to help build this!

---

*Meow is experimental research. Not production-ready.*